In [83]:
import pandas as pd
import glob
import string
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

Combining separte files into one

In [84]:
folder_path = "CSV files/*.csv"
all_files = glob.glob(folder_path)
df_list = [pd.read_csv(filename) for filename in all_files]
df = pd.concat(df_list, ignore_index=True)

In [85]:
df.head()

,,Time,Time.1,Place,Place.1,Place.2,Place.3,Place.4,Place.5,Place.6,...,Events.4,Events.5,Assessments,Assessments.1,Report 1,Report 1.1,Report 2,Report 2.1,Report 1.2,Unnamed: 125
0,ACN,Date,Local Time Of Day,Locale Reference,State Reference,Relative Position.Angle.Radial,Relative Position.Distance.Nautical Miles,Altitude.AGL.Single Value,Altitude.MSL.Single Value,Latitude / Longitude (UAS),...,When Detected,Result,Contributing Factors / Situations,Primary Problem,Narrative,Callback,Narrative,Callback,Synopsis,NaN
1,1714400,202001,0001-0600,ORD.Airport,IL,NaN,NaN,NaN,15000,NaN,...,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft; Company Policy,Company Policy,To be clear I did not have a lot going on and ...,NaN,NaN,NaN,A TRACON Departure Controller reported two air...,NaN
2,1714553,202001,1801-2400,ZHU.ARTCC,TX,NaN,NaN,NaN,14000,NaN,...,In-flight,Flight Crew Became Reoriented; General Physica...,Human Factors; Environment - Non Weather Related,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...,NaN,NaN,NaN,CRJ Captain reported that a laser shone repeat...,NaN
3,1714675,202001,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,9000,NaN,...,In-flight,Flight Crew Returned To Departure Airport; Fli...,Aircraft; Weather; Procedure,Weather,I was boarding the aircraft when a my destinat...,NaN,NaN,NaN,Pilot reported weather and a closed destinatio...,NaN
4,1714708,202001,0601-1200,ZZZ.Airport,US,NaN,NaN,0,NaN,NaN,...,In-flight,General None Reported / Taken,Procedure; Logbook Entry; Human Factors,Ambiguous,I had a ferry flight that went out of service ...,NaN,We departed ZZZ ferrying an un-airworthy aircr...,NaN,Air carrier Dispatcher and flight crew reporte...,NaN


In [86]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31141 entries, 0 to 31140
Columns: 126 entries,   to Unnamed: 125
dtypes: float64(1), str(125)
memory usage: 29.9 MB


In [87]:
df_1 = df.copy()

In [88]:
df_1 = df_1[['Assessments.1','Report 1']]

In [89]:
df_1.head()

,Assessments.1,Report 1
0,Primary Problem,Narrative
1,Company Policy,To be clear I did not have a lot going on and ...
2,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...
3,Weather,I was boarding the aircraft when a my destinat...
4,Ambiguous,I had a ferry flight that went out of service ...


In [90]:
df_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 31141 entries, 0 to 31140
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Assessments.1  31064 non-null  str  
 1   Report 1       31138 non-null  str  
dtypes: str(2)
memory usage: 486.7 KB


Checking for missing values

In [91]:
df_1.isnull().sum()/len(df_1) * 100

Assessments.1    0.247262
Report 1         0.009634
dtype: float64

Dropping Missing Values

In [92]:
df_1.dropna(inplace = True)

In [93]:
df_1.info()

<class 'pandas.DataFrame'>
Index: 31064 entries, 0 to 31140
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Assessments.1  31064 non-null  str  
 1   Report 1       31064 non-null  str  
dtypes: str(2)
memory usage: 728.1 KB


In [94]:
df_1.isnull().sum()/len(df_1) * 100

Assessments.1    0.0
Report 1         0.0
dtype: float64

Changing column name to first row

In [95]:
df_1.columns = df_1.iloc[0]
df_1 = df_1.drop(df.index[0])
df_1 = df_1.reset_index(drop=True)

In [96]:
df_1.head()

,Primary Problem,Narrative
0,Company Policy,To be clear I did not have a lot going on and ...
1,Environment - Non Weather Related,While descending into PIB out of 14;000 ft. MS...
2,Weather,I was boarding the aircraft when a my destinat...
3,Ambiguous,I had a ferry flight that went out of service ...
4,Aircraft,We were flying at 2500 ft. I started losing el...


In [97]:
df_1['Primary Problem'].value_counts()

Primary Problem
Human Factors                                         10823
Aircraft                                               9802
Procedure                                              2913
Ambiguous                                              2490
Weather                                                1059
Environment - Non Weather Related                       930
Airport                                                 786
Chart Or Publication                                    447
Airspace Structure                                      411
ATC Equipment / Nav Facility / Buildings                410
Company Policy                                          408
Software and Automation                                 177
Equipment / Tooling                                     103
Staffing                                                 96
MEL                                                      69
Incorrect / Not Installed / Unavailable Part             55
Manuals                 

Making narrative lowercase

In [98]:
df_1['Narrative'] = df_1['Narrative'].str.lower()
df_1['Narrative'][0]

"to be clear i did not have a lot going on and my complexity was not high. that is why this was a non-event. however; ord was super busy on two runways for a little bit and i am sure the tower was very busy landing 28c departing 28r and 22l. ground could have also been busy so that is why i put the information in there on complexity and amount of volume. here is what happened; we were on outboards at ord. there were a lot of aircraft going over the airport to balance the runways. i had a strip print and an aircraft depart and tag up. it was aircraft z. i was waiting for this aircraft to call on. i was waiting and waiting and making a plan. then the aircraft started climbing out of 5;000 feet; so i reached out and tried to turn the aircraft. he didn't turn or answer. i didn't know what happened or was happening. then i did it again. i was just about to call back to the tower and the position next to me was giving a briefing and they turned their aircraft to earnd.aircraft x. now they de

In [99]:
punc = string.punctuation

Removing punctuation

In [100]:
def remove_punc(text):
    return text.translate(str.maketrans('', '', punc))

In [101]:
df_1['Narrative'] = df_1['Narrative'].apply(remove_punc)

In [102]:
df_1['Narrative'][0]

'to be clear i did not have a lot going on and my complexity was not high that is why this was a nonevent however ord was super busy on two runways for a little bit and i am sure the tower was very busy landing 28c departing 28r and 22l ground could have also been busy so that is why i put the information in there on complexity and amount of volume here is what happened we were on outboards at ord there were a lot of aircraft going over the airport to balance the runways i had a strip print and an aircraft depart and tag up it was aircraft z i was waiting for this aircraft to call on i was waiting and waiting and making a plan then the aircraft started climbing out of 5000 feet so i reached out and tried to turn the aircraft he didnt turn or answer i didnt know what happened or was happening then i did it again i was just about to call back to the tower and the position next to me was giving a briefing and they turned their aircraft to earndaircraft x now they departed off of two diffe

Removing spelling mistakes from narrative

In [103]:
df_1['Narrative'] = df_1['Narrative'].apply(TextBlob)

Removing Stopwords

In [104]:
#nltk.download('stopwords')

In [105]:
stopword = stopwords.words('english')
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopword:
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [106]:
df_1['Narrative'] = df_1['Narrative'].apply(remove_stopwords)

In [107]:
df_1['Narrative'][0]

'  clear      lot going    complexity   high       nonevent however ord  super busy  two runways   little bit    sure  tower   busy landing 28c departing 28r  22l ground could  also  busy      put  information    complexity  amount  volume    happened    outboards  ord    lot  aircraft going   airport  balance  runways    strip print   aircraft depart  tag    aircraft z   waiting   aircraft  call    waiting  waiting  making  plan   aircraft started climbing   5000 feet   reached   tried  turn  aircraft  didnt turn  answer  didnt know  happened   happening           call back   tower   position next    giving  briefing   turned  aircraft  earndaircraft x   departed   two different runways   said   guy next   hey see    aircraft z  chance    somehow aircraft z going westbound departing   runway 28r     frequency  aircraft x departing   runway 22l   instruction  aircraft x  given aircraft z  aircraft x took   controller said  didnt hear  indication     step     anything   time aircraft  a

Dealing with excessive whitespaces

In [108]:
df_1['Narrative'] = df_1['Narrative'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [109]:
df_1['Narrative'][0]

'clear lot going complexity high nonevent however ord super busy two runways little bit sure tower busy landing 28c departing 28r 22l ground could also busy put information complexity amount volume happened outboards ord lot aircraft going airport balance runways strip print aircraft depart tag aircraft z waiting aircraft call waiting waiting making plan aircraft started climbing 5000 feet reached tried turn aircraft didnt turn answer didnt know happened happening call back tower position next giving briefing turned aircraft earndaircraft x departed two different runways said guy next hey see aircraft z chance somehow aircraft z going westbound departing runway 28r frequency aircraft x departing runway 22l instruction aircraft x given aircraft z aircraft x took controller said didnt hear indication step anything time aircraft also departed went incorrect frequency controller send correct frequency 3 similar sounding aircraft 1 frequency two supposed climbing two supposed tried explain 

In [110]:
#nltk.download('punkt')
#nltk.download('punkt_tab')

Tokenization

In [111]:
df_1['Narrative'] = df_1['Narrative'].apply(word_tokenize)

In [112]:
df_1['Narrative'][0]

['clear',
 'lot',
 'going',
 'complexity',
 'high',
 'nonevent',
 'however',
 'ord',
 'super',
 'busy',
 'two',
 'runways',
 'little',
 'bit',
 'sure',
 'tower',
 'busy',
 'landing',
 '28c',
 'departing',
 '28r',
 '22l',
 'ground',
 'could',
 'also',
 'busy',
 'put',
 'information',
 'complexity',
 'amount',
 'volume',
 'happened',
 'outboards',
 'ord',
 'lot',
 'aircraft',
 'going',
 'airport',
 'balance',
 'runways',
 'strip',
 'print',
 'aircraft',
 'depart',
 'tag',
 'aircraft',
 'z',
 'waiting',
 'aircraft',
 'call',
 'waiting',
 'waiting',
 'making',
 'plan',
 'aircraft',
 'started',
 'climbing',
 '5000',
 'feet',
 'reached',
 'tried',
 'turn',
 'aircraft',
 'didnt',
 'turn',
 'answer',
 'didnt',
 'know',
 'happened',
 'happening',
 'call',
 'back',
 'tower',
 'position',
 'next',
 'giving',
 'briefing',
 'turned',
 'aircraft',
 'earndaircraft',
 'x',
 'departed',
 'two',
 'different',
 'runways',
 'said',
 'guy',
 'next',
 'hey',
 'see',
 'aircraft',
 'z',
 'chance',
 'somehow',

Lemmitization

In [113]:
#nltk.download('wordnet')
#nltk.download('omw-1.4')

In [114]:
#nltk.download('averaged_perceptron_tagger', force=True)

In [115]:
lemmatizer = WordNetLemmatizer()

In [116]:
df_1['Narrative'] = df_1['Narrative'].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)

In [117]:
df_1['Narrative'][0]

['clear',
 'lot',
 'going',
 'complexity',
 'high',
 'nonevent',
 'however',
 'ord',
 'super',
 'busy',
 'two',
 'runway',
 'little',
 'bit',
 'sure',
 'tower',
 'busy',
 'landing',
 '28c',
 'departing',
 '28r',
 '22l',
 'ground',
 'could',
 'also',
 'busy',
 'put',
 'information',
 'complexity',
 'amount',
 'volume',
 'happened',
 'outboard',
 'ord',
 'lot',
 'aircraft',
 'going',
 'airport',
 'balance',
 'runway',
 'strip',
 'print',
 'aircraft',
 'depart',
 'tag',
 'aircraft',
 'z',
 'waiting',
 'aircraft',
 'call',
 'waiting',
 'waiting',
 'making',
 'plan',
 'aircraft',
 'started',
 'climbing',
 '5000',
 'foot',
 'reached',
 'tried',
 'turn',
 'aircraft',
 'didnt',
 'turn',
 'answer',
 'didnt',
 'know',
 'happened',
 'happening',
 'call',
 'back',
 'tower',
 'position',
 'next',
 'giving',
 'briefing',
 'turned',
 'aircraft',
 'earndaircraft',
 'x',
 'departed',
 'two',
 'different',
 'runway',
 'said',
 'guy',
 'next',
 'hey',
 'see',
 'aircraft',
 'z',
 'chance',
 'somehow',
 'a

Train test split

In [119]:
X = df_1['Narrative'].copy()
y = df_1['Primary Problem'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, random_state=42)